# Kaggle SFT runner (thin): any scale (sft_5k / sft_15k / sft = 50k), 3B, one T4

For each LR in `LRS`: **train** (Unsloth QLoRA, fp16) → push adapter + checkpoints to the HF Hub, curves to W&B → **merge** the adapter into the 16-bit base → **vLLM eval on dev** → **zip** → optional **checks** (vLLM run-to-run determinism on full dev, vLLM vs HF parity on 20 puzzles) → re-zip → dev metrics to the same W&B run.

**Run it as a background job, not an interactive session** (two interactive sessions died before the zip was written):
1. Settings: **Accelerator = GPU T4 x2**, **Internet = on**. Add-ons → Secrets: attach `HF_TOKEN` (write access) and `WANDB_API_KEY` (optional `WANDB_ENTITY`) to this notebook.
2. Edit the first cell (`SPLIT`, `LRS`, `COMMIT`), then **Save Version → Save & Run All (Commit)**. You can close the browser.
3. When the version finishes, download `sft_<run>.zip` from the version's **Output** tab. It is written right after the dev eval and rewritten after the checks.
4. **sft (150k) needs two sessions:** 11,554 steps × ~4.0–4.3 s ≈ 12.8–13.9 h of training, more than Kaggle's 12 h limit. Session 1 trains until Kaggle stops it (about 10,000 steps; a checkpoint is pushed every 100 steps), then session 2 runs with `RESUME = True` to finish, merge and evaluate.
5. If the run dies during training: set `RESUME = True` and commit again (continues from the Hub's `last-checkpoint/`). If it died after training: `EVAL_ONLY = True`.

Environments:
- **System Python = the eval env**, exactly as validated in `kaggle_eval.ipynb` (`requirements-gpu.txt` on top of the Kaggle image).
- **Training runs in a venv** (`/tmp/trainenv`, `--system-site-packages`). Unsloth's dependencies are installed into the venv and don't touch the eval env (known `datasets` / `huggingface-hub` conflict).

Every shell command goes through `run()`, which raises on a nonzero exit.

In [ ]:
# --- run settings ---
REPO = "https://github.com/twiggytom/tiny-gambits.git"
COMMIT = "main"            # pin to a commit hash for reported runs
SPLIT = "sft"              # sft_5k | sft_15k | sft_50k (done) | sft (= 150k, the final SFT model; needs 2 sessions)
LRS = [2e-4]               # frozen SFT LR (PLAN Open Items); the 5k sweep used [1e-4, 2e-4, 5e-4]
SEED = 0
CONFIG = "configs/sft.yaml"
CHECKS = True              # after the eval: vLLM determinism (full dev, 2nd run) + vLLM vs HF parity (20 puzzles)
GPU = "0"
RESUME = False             # True: continue the Hub repo's last-checkpoint/ (killed session)
EVAL_ONLY = False          # True: skip training; pull each run's final adapter from the Hub, merge, eval on dev
WANDB_RUN_IDS = {}         # run name → real W&B run id, when train_info.json recorded a wrong one
                           # (session 1: {"sft-5k-lr2e-4": "6yg4oe26"})

import json, os, shutil, subprocess, time
T_START = time.time()

def run(cmd: str) -> float:
    """Run a shell command, stream its output, raise if it fails. Returns wall-clock seconds."""
    print("$", cmd, flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")
    return time.time() - t

In [ ]:
run("nvidia-smi")

In [ ]:
os.chdir("/kaggle/working")
run(f"rm -rf tiny-gambits && git clone -q {REPO} tiny-gambits")
os.chdir("/kaggle/working/tiny-gambits")
run(f"git checkout -q {COMMIT} && git log -1 --format='%H %s'")
COMMIT_HASH = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

In [ ]:
# Eval env (system Python): same as kaggle_eval.ipynb.
run("pip install -q -r requirements-gpu.txt")
run("pip freeze > /kaggle/working/pip_freeze_eval.txt")
# Training env: venv on top of the system packages; unsloth's deps go into the venv only.
# --without-pip: Kaggle's Python has no ensurepip, so `python -m venv` fails without it. The venv's
# interpreter still sees the system pip (system site-packages), and pip run by the venv interpreter
# installs into the venv.
TRAIN_PY = "/tmp/trainenv/bin/python"
run("rm -rf /tmp/trainenv && python -m venv --system-site-packages --without-pip /tmp/trainenv")
run(f"{TRAIN_PY} -m pip --version")
run(f"{TRAIN_PY} -m pip install -q -r requirements-train.txt")
run(f"{TRAIN_PY} -m pip freeze > /kaggle/working/pip_freeze_train.txt")
run(f"{TRAIN_PY} -c \"import unsloth, torch, transformers, peft, trl; print('train env:', 'unsloth', unsloth.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, 'peft', peft.__version__, 'trl', trl.__version__); assert unsloth.__file__.startswith('/tmp/trainenv'), unsloth.__file__\"")
run("python -c \"import vllm, torch, transformers; print('eval env:', 'vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__)\"")
# The training install must not have changed the eval env.
run("pip freeze > /tmp/pip_freeze_eval_after.txt && diff /kaggle/working/pip_freeze_eval.txt /tmp/pip_freeze_eval_after.txt && echo 'eval env unchanged'")

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ["HF_TOKEN", "WANDB_API_KEY"]:
    os.environ[name] = secrets.get_secret(name)   # fails loudly if a secret is missing
    print(name, "loaded from Kaggle secrets")
try:  # optional: needed when the W&B account has no default entity
    os.environ["WANDB_ENTITY"] = secrets.get_secret("WANDB_ENTITY")
    print("WANDB_ENTITY loaded from Kaggle secrets")
except Exception:
    print("No WANDB_ENTITY secret; using the W&B account's default entity")
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
# Fail fast (before installs are wasted on a model load): the same W&B check train_sft runs first.
run("/tmp/trainenv/bin/python -c \"from src.train_sft import resolve_wandb_entity; print('W&B entity:', resolve_wandb_entity())\"")

In [ ]:
# Rebuild the SFT examples (downloads the pinned raw shards; ~1-2 min). Must match the local stats.
run(f"python -m src.sft_data --config configs/data.yaml --split {SPLIT} --seed 0")
stats = json.load(open(f"data/processed/{SPLIT}.stats.json"))
assert stats["n_examples"] == {"sft_5k": 12356, "sft_15k": 37026, "sft_50k": 123301, "sft": 369697}[SPLIT], stats["n_examples"]
SETUP_SECONDS = time.time() - T_START
print(f"setup: {SETUP_SECONDS / 60:.1f} min")

In [ ]:
import pandas as pd

def run_name(lr):
    out = subprocess.run([TRAIN_PY, "-c", f"from src.train_sft import run_name; print(run_name('{SPLIT}', {lr!r}))"],
                         capture_output=True, text=True, check=True)
    return out.stdout.strip()

def compare_outputs(ref_dir, other_dir):
    """Puzzle-by-puzzle comparison of model_outputs + outcome over the puzzles in other_dir."""
    a = pd.read_json(f"{ref_dir}/puzzles.jsonl", lines=True, dtype={"PuzzleId": str}).set_index("PuzzleId")
    b = pd.read_json(f"{other_dir}/puzzles.jsonl", lines=True, dtype={"PuzzleId": str}).set_index("PuzzleId")
    diff = [pid for pid in b.index if any(str(a.loc[pid, c]) != str(b.loc[pid, c]) for c in ["model_outputs", "outcome"])]
    return {"n": int(len(b)), "identical": int(len(b) - len(diff)),
            "differing": [{"PuzzleId": pid, "ref": list(a.loc[pid, "model_outputs"]), "other": list(b.loc[pid, "model_outputs"])}
                          for pid in diff]}

def zip_bundle(names):
    """results/{run} (+ raw check runs) + outputs/{run}/{train_info,timings}.json + pip freezes → /kaggle/working zip."""
    bundle = "/kaggle/working/bundle"
    shutil.rmtree(bundle, ignore_errors=True)
    for name in names:
        shutil.copytree(f"results/{name}", f"{bundle}/results/{name}")
        for raw in [f"results/raw/{name}_vllm_rerun", f"results/raw/{name}_parity_hf"]:
            if os.path.isdir(raw):
                shutil.copytree(raw, f"{bundle}/{raw}")
        os.makedirs(f"{bundle}/outputs/{name}", exist_ok=True)
        for fn in ["train_info.json", "timings.json"]:
            if os.path.exists(f"outputs/{name}/{fn}"):
                shutil.copy(f"outputs/{name}/{fn}", f"{bundle}/outputs/{name}/{fn}")
    for fn in ["pip_freeze_eval.txt", "pip_freeze_train.txt"]:
        shutil.copy(f"/kaggle/working/{fn}", f"{bundle}/{fn}")
    path = shutil.make_archive(f"/kaggle/working/sft_{'_'.join(names)}", "zip", bundle)
    print("zip written:", path, flush=True)
    return path

In [ ]:
done = []
for lr in LRS:
    name = run_name(lr)
    merged = f"/tmp/merged/{name}"
    print(f"\n===== {name} =====", flush=True)
    if EVAL_ONLY:   # adapter + train_info.json from the Hub; training is not repeated
        t_train = None
        run(f"{TRAIN_PY} -m src.merge_adapter --config {CONFIG} --run-name {name} --out {merged} --from-hub")
    else:
        resume = "--resume" if RESUME else ""
        t_train = run(f"{TRAIN_PY} -m src.train_sft --config {CONFIG} --split {SPLIT} --lr {lr} --seed {SEED} --merge {merged} {resume}")
    dev = f"results/{name}/dev"
    t_eval = run(f"python -m src.evaluate --config configs/eval.yaml --model {merged} --backend vllm --split dev --run-name {name} --seed {SEED}")
    info = json.load(open(f"outputs/{name}/train_info.json"))
    timings = {"setup_seconds": SETUP_SECONDS, "train_process_seconds": t_train,
               "merge_seconds": info["merge_seconds"], "eval_seconds": t_eval, "commit": COMMIT_HASH,
               "eval_only": EVAL_ONLY}
    json.dump(timings, open(f"outputs/{name}/timings.json", "w"), indent=2)
    run(f"{TRAIN_PY} -m src.project_compute --config {CONFIG} --train-info outputs/{name}/train_info.json --timings outputs/{name}/timings.json")
    done.append(name)
    zip_bundle(done)   # the main result is saved before anything optional runs

    # Optional checks: mismatches are recorded, never raised; a crash here doesn't lose the result above.
    if CHECKS:
        checks = {}
        try:
            t = run(f"python -m src.evaluate --config configs/eval.yaml --model {merged} --backend vllm --split dev --run-name raw/{name}_vllm_rerun --seed {SEED}")
            checks["vllm_determinism"] = {**compare_outputs(dev, f"results/raw/{name}_vllm_rerun/dev"), "seconds": t}
            t = run(f"python -m src.evaluate --config configs/eval.yaml --model {merged} --backend hf --split dev --limit 20 --run-name raw/{name}_parity_hf --seed {SEED}")
            checks["vllm_hf_parity"] = {**compare_outputs(dev, f"results/raw/{name}_parity_hf/dev"), "seconds": t,
                                        "hf_dtype": json.load(open(f"results/raw/{name}_parity_hf/dev/run_info.json"))["dtype"]}
        except Exception as e:
            checks["error"] = repr(e)
            print(f"WARNING: checks failed for {name}: {e}")
        json.dump(checks, open(f"{dev}/checks.json", "w"), indent=2)
        for k, v in checks.items():
            print(k, "error" if k == "error" else f"{v['identical']}/{v['n']} identical", v if k == "error" else [d["PuzzleId"] for d in v["differing"]])
        zip_bundle(done)
    shutil.rmtree(merged)   # ~6 GB each; the adapter is on the Hub

    # W&B is a mirror; the zipped files are the record. A W&B failure must not stop the run.
    try:
        run(f"{TRAIN_PY} -m src.wandb_log --train-info outputs/{name}/train_info.json --metrics {dev}/metrics.json --timings outputs/{name}/timings.json"
            + (f" --run-id {WANDB_RUN_IDS[name]}" if name in WANDB_RUN_IDS else ""))
    except RuntimeError as e:
        print(f"WARNING: W&B eval logging failed for {name} (results are still saved and zipped): {e}")

In [ ]:
# Checks + summary. T4 must train and eval in fp16; the adapter must be on the Hub.
from huggingface_hub import list_repo_files
rows = []
for name in done:
    info = json.load(open(f"outputs/{name}/train_info.json"))
    ev = json.load(open(f"results/{name}/dev/run_info.json"))
    m = json.load(open(f"results/{name}/dev/metrics.json"))
    for what, rec in [("train", info), ("eval", ev)]:
        assert "T4" not in rec["gpu"] or rec["dtype"] == "float16", f"{name} {what}: T4 must use fp16, got {rec['dtype']}"
    assert EVAL_ONLY or info["n_steps"] == info["expected_steps"], (info["n_steps"], info["expected_steps"])
    files = list_repo_files(info["hub_repo"])
    assert "adapter_model.safetensors" in files and any(f.startswith("last-checkpoint/") for f in files), files
    rows.append({"run": name, "steps": info["n_steps"], "mean loss": round(info.get("mean_logged_loss") or info["final_train_loss"], 4),
                 "s/step": round(info.get("seconds_per_step_all_processes") or info["seconds_per_step"], 2),
                 "train h (all sessions)": round((info.get("train_seconds_all_processes") or info["train_seconds"]) / 3600, 2), "peak VRAM GB": round(info["peak_vram_gb"]["reserved"], 2),
                 "first_move": m["first_move_acc"], "full_solution": m["full_solution_acc"],
                 "illegal": m["illegal_rate"], "unparseable": m["unparseable_rate"],
                 "distinct_first_moves": len({json.loads(l)["model_outputs"][0] for l in open(f"results/{name}/dev/puzzles.jsonl")})})
    if os.path.exists(f"results/{name}/dev/checks.json"):
        c = json.load(open(f"results/{name}/dev/checks.json"))
        rows[-1].update({k: f"{v['identical']}/{v['n']}" for k, v in c.items() if k != "error"})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Final zip (same contents as the per-run zips; written again in case the summary is the last thing to run).
zip_bundle(done)